# 03 — Silver Transformations
### Goal
- import reusable code from src/
- use df.transform()
- apply one UDF
- write silver Delta tables

In [0]:
%run ../01_setup/00_config

In [0]:
import sys

sys.path.append(f"{repo_path}/src")

from transforms.market_prices_cleaning import standardize_market_prices_columns, filter_invalid_market_prices_rows
from transforms.business_rules import add_market_prices_flags
from udfs.market_prices_udfs import classify_price_band

In [0]:
bronze_df = spark.table(bronze_market_prices_table)

silver_df = (
    bronze_df
    .transform(standardize_market_prices_columns)
    .transform(lambda df: filter_invalid_market_prices_rows(df, rules))
    .transform(lambda df: add_market_prices_flags(df, rules))
    .withColumn("price_band", classify_price_band(F.col("price_eur_mwh")))
)

display(silver_df.limit(10))
print(f"Bronze rows: {bronze_df.count()} - Silver rows: {silver_df.count()}")


In [0]:
silver_df.write.mode("overwrite").saveAsTable(silver_market_prices_table)
print(f"Silver table saved: {silver_market_prices_table}")

In [0]:
dbutils.library.restartPython()